In [ ]:
# =========================
# Standard Libraries
# =========================
import numpy as np
import pandas as pd

# =========================
# Visualization
# =========================
import matplotlib.pyplot as plt
import seaborn as sns
import shap

# =========================
# Sklearn Core
# =========================
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.calibration import CalibratedClassifierCV

# =========================
# Models
# =========================
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import RandomizedSearchCV

# Boosting Models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# =========================
# Evaluation Metrics
# =========================
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    precision_recall_curve,
    roc_curve,
    accuracy_score,
    f1_score,
    average_precision_score,
    recall_score, 
    precision_score,
    accuracy_score,
    precision_recall_curve
)

# =========================
# Utilities
# =========================
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

RANDOM_STATE = 42

In [ ]:
# function for data preparation and cleaning
def read_data(fname: str) -> pd.DataFrame:
    # read data
    data = pd.read_csv(fname)

    # Basic checks
    print(f"Data shape raw: {data.shape}")
    
    n_duplicates = data.duplicated().sum()
    print(f"Number of duplicates: {n_duplicates}")

    data = data.drop_duplicates(keep = "last")

    # target encoding
    data["Result"] = data["Result"].str.lower().map({"negative": 0, "positive": 1})

    if data["Result"].isnull().any():
        raise ValueError("Unexpected values in target column")
    
    
    print(f"\nData shape after dropping: \n{data.shape}")

    print("\nData types:\n", data.dtypes)
    
    print("\nMissing values:\n", data.isnull().sum())

    data.insert(0, "patient_id", range(1, len(data)+1))
    data.set_index("patient_id", inplace=True)

    return data

In [ ]:
# Read the data
data = read_data(fname='Medicaldataset.csv')

In [ ]:
# Checking data types
data.info()

In [ ]:
# checking data basic statistical description
data.describe()

In [ ]:
# data preprocessing

# input - output splitting process
def split_input_output(data: pd.DataFrame, target_col: str):
    # Validate column existence
    if target_col not in data.columns:
        raise ValueError(f"{target_col} not found in dataframe")

    # Warning if target has missing values
    if data[target_col].isna().sum() > 0:
        print("Warning: Target column contains missing values")

    # Separate input (X) and output (y)
    X = data.drop(columns=[target_col])
    y = data[target_col]

    # Print shapes after splitting
    print(f"X shape: {X.shape}")
    print(f"y shape: {y.shape}")

    # Print class distribution
    print("Target distribution:\n", data[target_col].value_counts(normalize=True))
    print(y.value_counts())

    return X, y

In [ ]:
# Load the train data only
X, y = split_input_output(data=data,
                          target_col='Result')

In [ ]:
# checking input variables
X.head()

In [ ]:
# checking output variable
y.head()

In [ ]:
# train - validation - test splitting process
# First split: Train vs Not Train
X_train, X_not_train, y_train, y_not_train = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

# Second split: Validation vs Test
X_valid, X_test, y_valid, y_test = train_test_split(
    X_not_train,
    y_not_train,
    test_size=0.5,
    random_state=RANDOM_STATE,
    stratify=y_not_train
)

print(f"X train shape: {X_train.shape}")
print(f"y train shape: {y_train.shape}")

print(f"X valid shape: {X_valid.shape}")
print(f"y valid shape: {y_valid.shape}")

print(f"X test shape: {X_test.shape}")
print(f"y test shape: {y_test.shape}")

In [ ]:
# Validate the data
print(len(X_train)/len(X))  # should be 0.8
print(len(X_valid)/len(X))  # should be 0.1
print(len(X_test)/len(X))   # should be 0.1

In [ ]:
# checking input data train
X_train.head()

In [ ]:
# checking output data train
y_train.head()

In [ ]:
# Checking missing values of the data train
missing_percent = X_train.isna().mean() * 100
print(missing_percent)

In [ ]:
# Finding anomalous data
X_train.describe()

In [ ]:
'''
Heart rate anomalous sign :
    value > 220 : 
        According to medical research of journal of the american college of cardiology,
        maximum heart rate is often estimated using the age predicted equation of 220 - age.

        The minimum age on this medical dataset is 14

        Hence, maximum of heart rate in this dataset should be :
            220 - 14 = 206

        otherwise, it would be a sign of anomalous data 
'''
X_train[X_train['Heart rate']>=206]

In [ ]:
# removing anomalous data

def remove_invalid_hr(X, y):
    cond = (X['Heart rate'] > 206)
    idx_to_drop = X[cond].index

    X = X.drop(index=idx_to_drop)
    y = y.drop(index=idx_to_drop)

    return X, y


X_train, y_train = remove_invalid_hr(X_train, y_train)
X_valid, y_valid = remove_invalid_hr(X_valid, y_valid)
X_test, y_test = remove_invalid_hr(X_test, y_test)

In [ ]:
# Data Exploration (EDA)

# Concat the data first
train_data = pd.concat((X_train, y_train), axis=1)
train_data.head()

In [ ]:
# checking balance condition data train of result output using bar chart
sns.countplot(x=y_train)
plt.title("Heart Attack")
plt.show()

In [ ]:
# Plot histogram Only Numeric Columns

numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
n_cols = 3
n_rows = int(np.ceil(len(numeric_cols) / n_cols))

fig, ax = plt.subplots(nrows=n_rows, ncols=n_cols, figsize=(12, 4*n_rows))
axes = ax.flatten()

for i, col in enumerate(numeric_cols):
    sns.kdeplot(X_train[col], ax=axes[i])
    axes[i].set_title(f'Distribution of {col}')

# remove unused axes
for j in range(len(numeric_cols), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix of the input variables 

train_numeric = X_train.select_dtypes(include=["int64","float64"])

corr = train_numeric.corr()

plt.figure(figsize=(12,8))
sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

In [ ]:
# Correlation values of the input variables to the output
corr_target = train_numeric.join(y_train).corr()["Result"].sort_values(ascending=False)
print(corr_target)

In [ ]:
# Implement Preprocessing Pipeline

# Identify column types
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

# Define log-transformed features
log_features = ["CK-MB", "Troponin"]

# Remove log features from numeric features to avoid duplication
numeric_features = [col for col in numeric_features if col not in log_features]

print("Numeric Features:", numeric_features)
print("Log Features:", log_features)
print("Categorical Features:", categorical_features)

# Pipelines
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

log_transformer = FunctionTransformer(np.log1p)

# ColumnTransformer
preprocessor = ColumnTransformer([
    ("log", log_transformer, log_features),
    ("num", numeric_transformer, numeric_features)
])

In [ ]:
'''
Defining Evaluation Metrics

Heart Attack Occurrence Confusion Matrix

| Reality      | Predicted 0 (No Heart Attack) | Predicted 1 (Heart Attack)  |
|--------------|-------------------------------|-----------------------------|
| 0 (No)       | True Negative (Correct)       | False Positive              |
| 1 (Yes)      | False Negative                | True Positive (Correct)     |

'''

In [ ]:
'''
Define Baseline Model

Baseline Model:
    Logistic Regression
    
'''

In [ ]:
# =========================
# BASELINE MODEL TRAINING
# =========================

y_train = y_train.astype(int)
y_test = y_test.astype(int)

baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",   # handle class imbalance
        random_state=42,           # reproducibility
        max_iter=1000              # ensure convergence
    ))
])

# Train model
baseline_model.fit(X_train, y_train)

# =========================
# MODEL PREDICTIONS
# =========================

# Class predictions
y_pred_base = baseline_model.predict(X_test)

# Probability predictions (for ROC-AUC)
y_proba_base = baseline_model.predict_proba(X_test)[:, 1]

# =========================
# MODEL EVALUATION
# =========================

print("Baseline Model - Logistic Regression\n")

print("Classification Report:\n")
print(classification_report(y_test, y_pred_base))

print("ROC-AUC Score:", roc_auc_score(y_test, y_proba_base))

In [ ]:
# Storing models to a list

models = [
    ("Logistic Regression", LogisticRegression(max_iter=500)),
    ("KNN", KNeighborsClassifier(n_neighbors=10)),
    ("RandomForest", RandomForestClassifier(random_state=123)),
    ("SVM", SVC()),
    ("XGBoost", XGBClassifier())
]

# =========================
# LIST FOR STORING RESULTS
# =========================
precision_result = []
acc_result = []
model_names = []

# Anti-duplicate sets
prec_seen = set()
acc_seen = set()
name_seen = set()

# =========================
# CLEAR LIST BEFORE TRAINING
# =========================
from warnings import filterwarnings


In [ ]:
filterwarnings('ignore')

model_names.clear()
precision_result.clear()
acc_result.clear()
name_seen.clear()
prec_seen.clear()
acc_seen.clear()

# =========================
# TRAIN & EVALUATE MODELS
# =========================
from sklearn.metrics import precision_score, accuracy_score, classification_report

for name, model in models:
    
    # Fit model
    model.fit(X_train, y_train)
    
    # Predict
    y_pred_val = model.predict(X_valid)
    
    # Metrics
    score = precision_score(y_valid, y_pred_val, average='macro')
    acc_score = accuracy_score(y_valid, y_pred_val)
    
    # Store precision (avoid duplicate)
    if score not in prec_seen:
        prec_seen.add(score)
        precision_result.append(score)
    
    # Store accuracy
    acc_result.append(acc_score)
    
    # Store model name (avoid duplicate)
    if name not in name_seen:
        name_seen.add(name)
        model_names.append(name)
    
    # Print report
    print(f"Model: {name}")
    print(classification_report(
        y_valid,
        y_pred_val,
        target_names=["Negative (0)", "Positive (1)"]
    ))

In [ ]:
# Storing models to a list

models = [
    ("Logistic Regression", LogisticRegression(max_iter=500)),
    ("KNN", KNeighborsClassifier(n_neighbors=10)),
    ("RandomForest", RandomForestClassifier(random_state=123)),
    ("SVM", SVC()),
    ("XGBoost", XGBClassifier())
]

# =========================
# LIST FOR STORING RESULTS
# =========================
precision_result = []
acc_result = []
model_names = []

# Anti-duplicate sets
prec_seen = set()
acc_seen = set()
name_seen = set()


In [ ]:
model_names.clear()
precision_result.clear()
acc_result.clear()
name_seen.clear()
prec_seen.clear()
acc_seen.clear()

# =========================
# TRAIN & EVALUATE MODELS
# =========================

results = []

for name, model in models:
    
    model.fit(X_train, y_train)
    y_pred_val = model.predict(X_valid)
    
    results.append({
        "Model": name,
        "Precision": precision_score(y_valid, y_pred_val),
        "Recall": recall_score(y_valid, y_pred_val),
        "Accuracy": accuracy_score(y_valid, y_pred_val)
    })

precision_table = (
    pd.DataFrame(results)
    .sort_values(by="Recall", ascending=False)
    .reset_index(drop=True)
)

precision_table

In [ ]:
# Hyperparameter Tuning

# function to evaluate model
def evaluate(model, X, y):
    y_pred = model.predict(X)

    precision = precision_score(y, y_pred)
    recall = recall_score(y, y_pred)
    accuracy = accuracy_score(y, y_pred)

    print("Model Performance")
    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")

    return recall

In [ ]:
# =========================
# Random Forest
# =========================

# Base model
rf_tree = RandomForestClassifier(random_state=123)

# =========================
# Hyperparameters
# =========================
params = {
    'n_estimators': [100, 200, 500, 700],
    'max_features': ['sqrt', 'log2'],   # 'auto' is deprecated
    'min_samples_leaf': [2, 3, 4, 5, 10, 15, 20],
    'min_samples_split': [2, 4, 6, 8, 10, 12, 15, 20],
    'max_depth': list(range(1, 31)),
    'bootstrap': [True, False]
}

# =========================
# Random Search
# =========================
rf_tree_cv = RandomizedSearchCV(
    estimator=rf_tree,
    param_distributions=params,
    cv=10,
    scoring='f1',   # ✅ better for medical use
    n_iter=50,
    n_jobs=-1,
    random_state=123
)

# =========================
# Fit tuned model
# =========================
rf_tree_cv.fit(X_train, y_train)

print("Best RF:", rf_tree_cv.best_estimator_)

In [ ]:
# =========================
# Base model
# =========================
rf_base = rf_tree
rf_base.fit(X_train, y_train)

rf_base_result = evaluate(rf_base, X_valid, y_valid)

print("\n--- After hyperparameter tuning ---\n")

# =========================
# Tuned model
# =========================
rf_tuned = rf_tree_cv.best_estimator_

rf_tuned_result = evaluate(rf_tuned, X_valid, y_valid)

# =========================
# Improvement
# =========================
improvement = 100 * (rf_tuned_result - rf_base_result) / rf_base_result

print(f"Recall improvement: {improvement:.2f}%")

In [ ]:
# =========================
# XGBoost PIPELINE
# =========================

xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        random_state=123,
        eval_metric='logloss'
    ))
])

# =========================
# Hyperparameters
# =========================
params = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [3, 5, 7],
    "classifier__learning_rate": [0.05, 0.1, 0.2]
}

# =========================
# Random Search
# =========================
xgb_cv = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=params,
    cv=5,
    scoring='recall',
    n_iter=10,
    n_jobs=-1,
    random_state=123
)

# =========================
# Fit
# =========================
xgb_cv.fit(X_train, y_train)

print("Best XGBoost Pipeline:")
print(xgb_cv.best_estimator_)

In [ ]:
# =========================
# Base model
# =========================
xgb_base = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        random_state=123,
        eval_metric='logloss'
    ))
])

xgb_base.fit(X_train, y_train)
xgb_base_result = evaluate(xgb_base, X_valid, y_valid)

print("\n--- After hyperparameter tuning ---\n")

# =========================
# Tuned model
# =========================
xgb_tuned = xgb_cv.best_estimator_

xgb_tuned_result = evaluate(xgb_tuned, X_valid, y_valid)


# =========================
# Improvement
# =========================
improvement = 100 * (xgb_tuned_result - xgb_base_result) / xgb_base_result

print(f"Recall improvement: {improvement:.2f}%")

In [ ]:
# =========================
# 5. Final Model
# =========================

# Based on evaluation, XGBoost is selected
xgb_final = xgb_cv.best_estimator_

print("Final Model:")
print(xgb_final)

In [ ]:
# =========================
# Evaluate on Test Data (XGBoost)
# =========================

y_pred_xgb_final = xgb_final.predict(X_test)

print(classification_report(
    y_test,
    y_pred_xgb_final,
    target_names=["Negative (0)", "Positive (1)"]
))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_xgb_final)

In [ ]:
y_test.value_counts()

In [ ]:
# TP / (TP + FN)
recall_class1 = 80 / (80 + 1)
print(recall_class1)

In [ ]:
# =========================
# ROC Curve - XGBoost
# =========================

# Use probabilities (IMPORTANT)
y_proba_xgb = xgb_final.predict_proba(X_test)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_proba_xgb)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label="XGBoost")
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - XGBoost")
plt.legend()
plt.show()

auc_score = roc_auc_score(y_test, y_proba_xgb)
print("ROC-AUC:", auc_score)

In [ ]:
# =========================
# REBUILD FEATURE NAMES
# =========================
log_features = ["CK-MB", "Troponin"]

numeric_features = [
    col for col in X_train.columns
    if col not in log_features
]

feature_names = log_features + numeric_features

# =========================
# TRANSFORM DATA (USE FITTED PIPELINE)
# =========================
X_test_transformed = xgb_final.named_steps["preprocessor"].transform(X_test)

X_test_df = pd.DataFrame(
    X_test_transformed,
    columns=feature_names
)

# =========================
# SHAP EXPLANATION
# =========================
model = xgb_final.named_steps["classifier"]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_df)

# =========================
# PLOT
# =========================
shap.summary_plot(shap_values, X_test_df)

In [ ]:
'''
SHAP analysis confirms that Troponin and CK-MB are the most influential features driving model predictions,
aligning with established clinical knowledge.
'''

In [ ]:
# Dump Model to pkl file

filename = xgb_final_model.pkl

joblib.dump(xgb_pipeline, "final_model.pkl")